### Missing values in the data

should they just be removed?

In [48]:
%load_ext autoreload
%autoreload 2
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt



BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2025
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"

df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# --- 3. standardize to hourly, then daily ---

df_prices_hourly  = df_prices.resample("h").mean()
df_flows_hourly   = df_flows.resample("h").mean()
df_system_hourly  = df_system.resample("h").mean()

df_flows_hourly = df_flows_hourly.fillna(0)


# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_hourly.add_prefix("price_"),
    df_flows_hourly.add_prefix("flow_"),
    df_system_hourly.add_prefix("system_"),
], axis=1).sort_index()

print(f"\nMissing values:\n{df_daily.isna().sum()}")
# Fill missing flow values with 0 (no flow = connection didn't exist yet or was down)
flow_cols = [col for col in df_daily.columns if col.startswith("flow_")]
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)

# Drop rows where everything else is still missing
df_daily = df_daily.dropna(how="all")

# Optional: check what's left
print(f"Shape after cleaning: {df_daily.shape}")
print(f"Remaining missing values:\n{df_daily.isna().sum()[df_daily.isna().sum() > 0]}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:102: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()
/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:102: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()



Missing values:
price_ee                              0
price_fi                              0
price_lv                              0
price_lt                              0
flow_('ee', 'fi')                     1
flow_('ee', 'lv')                     1
flow_('ee', 'ru_narva')               1
flow_('ee', 'ru_pihkva')              1
system_production                    18
system_consumption                   21
system_losses                     62113
system_frequency                      1
system_system_balance                15
system_ac_balance                    15
system_production_renewable          19
system_solar_energy_production     8767
dtype: int64
Shape after cleaning: (62113, 16)
Remaining missing values:
system_production                    18
system_consumption                   21
system_losses                     62113
system_frequency                      1
system_system_balance                15
system_ac_balance                    15
system_production_renewable   

In [49]:
print(df_daily.columns.tolist())

['price_ee', 'price_fi', 'price_lv', 'price_lt', "flow_('ee', 'fi')", "flow_('ee', 'lv')", "flow_('ee', 'ru_narva')", "flow_('ee', 'ru_pihkva')", 'system_production', 'system_consumption', 'system_losses', 'system_frequency', 'system_system_balance', 'system_ac_balance', 'system_production_renewable', 'system_solar_energy_production']


In [50]:
print("Raw df_flows columns:", df_flows.columns.tolist())
print("After resample columns:", df_flows_hourly.columns.tolist())

Raw df_flows columns: [('ee', 'fi'), ('ee', 'lv'), ('ee', 'ru_narva'), ('ee', 'ru_pihkva')]
After resample columns: [('ee', 'fi'), ('ee', 'lv'), ('ee', 'ru_narva'), ('ee', 'ru_pihkva')]


### Mean, std 

In [51]:



# ==================================================
# ST-GNN: ESTONIAN ENERGY RESILIENCE MODEL
# Target: True energy balance (production + imports - consumption)
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)
# ==================================================


prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()

idx      = prices_h.index
flows_h  = flows_h.reindex(idx, method="ffill")
system_h = system_h.reindex(idx, method="ffill")

# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["gross_supply_input"] = (
    system_h["production"]
    - flows_h[("ee", "fi")] #negative flows are imports, positive flows are exports, so we subtract the imports
    - flows_h[("ee", "lv")] - flows_h[("ee", "ru_narva")] - flows_h[("ee", "ru_pihkva")]
)

print(f"    Mean: {system_h['gross_supply_input'].mean():+.1f} MW")
print(f"    Std:  {system_h['gross_supply_input'].std():.1f} MW")
print(f"    Min:  {system_h['gross_supply_input'].min():+.1f} MW")
print(f"    Max:  {system_h['gross_supply_input'].max():+.1f} MW")
print(f"    True deficit hours: {(system_h['gross_supply_input'] < 0).sum()}")


    Mean: +928.7 MW
    Std:  198.0 MW
    Min:  +412.6 MW
    Max:  +1600.1 MW
    True deficit hours: 0


In [52]:
flows_h.columns.tolist()
flows_h

,"(ee, fi)","(ee, lv)","(ee, ru_narva)","(ee, ru_pihkva)"
timestamp,,,,
2019-01-01 00:00:00+00:00,125.7966,119.0860,-165.2681,-10.4020
2019-01-01 01:00:00+00:00,35.8953,157.2594,-118.5069,-0.3477
2019-01-01 02:00:00+00:00,70.3056,122.3419,-135.8722,9.0883
2019-01-01 03:00:00+00:00,60.0575,117.6249,-132.6681,14.6492
2019-01-01 04:00:00+00:00,131.9497,119.3417,-190.1111,0.2314
...,...,...,...,...
2026-01-31 20:00:00+00:00,-999.7225,386.6525,0.0000,0.0000
2026-01-31 21:00:00+00:00,-995.2117,376.0808,0.0000,0.0000
2026-01-31 22:00:00+00:00,-988.6600,305.9417,0.0000,0.0000


EE-FI mean: -624 MW  → Estonia imports ~624 MW from Finland on average

EE-LV mean: +249 MW  → Estonia exports ~249 MW to Latvia on average

In [ ]:
flows_h[("ee", "fi")].describe()

count    62113.000000
mean      -624.460111
std        373.506405
min      -1006.658300
25%       -980.483300
50%       -721.183300
75%       -349.475000
max       1033.116000
Name: (ee, fi), dtype: float64

In [ ]:
flows_h[("ee", "lv")].describe()

count    62113.000000
mean       248.605089
std        271.094887
min       -820.216700
25%         88.733300
50%        271.600000
75%        441.825000
max        898.666700
Name: (ee, lv), dtype: float64

In [54]:
flows_h[("ee", "ru_narva")].describe()

count    62113.000000
mean        60.788663
std        204.660148
min       -726.750000
25%        -51.583300
50%         33.333300
75%        197.233300
max        901.016700
Name: (ee, ru_narva), dtype: float64

In [55]:
flows_h[("ee", "ru_pihkva")].describe()

count    62113.000000
mean        18.887527
std         51.758048
min       -273.525000
25%          0.000000
50%          0.000000
75%         45.491700
max        339.716700
Name: (ee, ru_pihkva), dtype: float64